# Elgiganten

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [3]:
url = "https://www.elgiganten.se/datorer-kontor/datorer"
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.14; rv:65.0) Gecko/20100101 Firefox/65.0"
}

response = requests.get(url, headers=headers)
print("Status code:", response.status_code)

Status code: 200


In [4]:
print("Scraping all links for every computer from Elgiganten")

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {"User-Agent": "Mozilla/5.0"}
base_url = "https://www.elgiganten.se"
start_url = base_url + "/datorer-kontor/datorer"

all_links = set()
current_url = start_url

while current_url:
    print(f"Scraping: {current_url}")
    response = requests.get(current_url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # Get all <a> tags with data-testid="product-card"
    for tag in soup.find_all("a", {"data-testid": "product-card"}):
        href = tag.get("href")
        if href and href.startswith("/product/datorer-kontor/datorer/laptop/"):
            full_url = base_url + href
            all_links.add(full_url)

    # Find the next page link
    next_tag = soup.find("a", rel="next")
    if next_tag and next_tag.get("href"):
        current_url = base_url + next_tag["href"]
        time.sleep(1)
    else:
        current_url = None  # No more pages

# Save to Excel
df = pd.DataFrame(sorted(all_links), columns=["Laptop Link"])
df.to_excel("elgiganten_laptop_links.xlsx", index=False)
print("Done! Saved to elgiganten_laptop_links.xlsx")

Scraping all links for every computer from Elgiganten
Scraping: https://www.elgiganten.se/datorer-kontor/datorer
Scraping: https://www.elgiganten.se/datorer-kontor/datorer/page-2
Scraping: https://www.elgiganten.se/datorer-kontor/datorer/page-3
Scraping: https://www.elgiganten.se/datorer-kontor/datorer/page-4
Scraping: https://www.elgiganten.se/datorer-kontor/datorer/page-5
Scraping: https://www.elgiganten.se/datorer-kontor/datorer/page-6
Scraping: https://www.elgiganten.se/datorer-kontor/datorer/page-7
Scraping: https://www.elgiganten.se/datorer-kontor/datorer/page-8
Done! Saved to elgiganten_laptop_links.xlsx


In [5]:
import requests
from bs4 import BeautifulSoup

print("Testing scraping name, price, specs and links from one cpomputer")
url = "https://www.elgiganten.se/product/datorer-kontor/datorer/laptop/acer-aspire-3-cel4128-173-barbar-dator/767608"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

# Product Name
name_tag = soup.find("h1")
product_name = name_tag.get_text(strip=True) if name_tag else "N/A"

# Price
price_tag = soup.select_one("span.inc-vat")
price = price_tag.get_text(strip=True) if price_tag else "N/A"

# Specs: find <h2> containing "Tekniska specifikationer"
specs = []
h2_tag = soup.find("h2", string=lambda text: text and "Teknisk specifikation" in text)
if h2_tag:
    ul_tag = h2_tag.find_next("ul")
    if ul_tag:
        for li in ul_tag.find_all("li"):
            specs.append(li.get_text(strip=True))

# Output
print("Product:", product_name)
print("Price:", price)
print("Specs:")
for spec in specs:
    print("   -", spec)

Testing scraping name, price, specs and links from one cpomputer
Product: Acer Aspire 3 Cel/4/128 17.3" bärbar dator
Price: 4996.-
Specs:
   - Intel® Celeron® N100 processor
   - 17.3" HD+ TN-skärm
   - 4 GB DDR5 RAM, 128 GB eMMC-lagring


In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time


print("From the .xlsx file with the links of all the computers scraping the name, price, specs and links from all the computers")
# Load Excel file with links
input_file = "elgiganten_laptop_links.xlsx"  # make sure this file is in the same folder
df = pd.read_excel(input_file)

headers = {"User-Agent": "Mozilla/5.0"}
results = []

for index, row in df.iterrows():
    url = row['Laptop Link']
    print(f"Scraping {index+1}/{len(df)}: {url}")
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.text, "html.parser")

        # Name
        name_tag = soup.find("h1")
        name = name_tag.get_text(strip=True) if name_tag else "N/A"

        # Price
        price_tag = soup.select_one("span.inc-vat")
        price = price_tag.get_text(strip=True).replace(".-", "") if price_tag else "N/A"

        # Specs
        specs = []
        h2_tag = soup.find("h2", string=lambda t: t and "Teknisk specifikation" in t)
        if h2_tag:
            ul = h2_tag.find_next("ul")
            if ul:
                specs = [li.get_text(strip=True) for li in ul.find_all("li")]

        # Append result
        results.append({
            "Name": name,
            "Price": price,
            "Specs": " | ".join(specs),
            "URL": url
        })

    except Exception as e:
        print(f"Error on {url}: {e}")
        results.append({
            "Name": "Error",
            "Price": "Error",
            "Specs": "Error",
            "URL": url
        })

    time.sleep(1)  # pause

# Save to Excel
output_df = pd.DataFrame(results)
output_df.to_excel("elgiganten_laptops_full.xlsx", index=False)

print("All done! Saved to 'elgiganten_laptops_full.xlsx'")

From the .xlsx file with the links of all the computers scraping the name, price, specs and links from all the computers
Scraping 1/331: https://www.elgiganten.se/product/datorer-kontor/datorer/laptop/acer-aspire-3-cel4128-173-barbar-dator/767608
Scraping 2/331: https://www.elgiganten.se/product/datorer-kontor/datorer/laptop/acer-aspire-3-i3-n3058128-173-barbar-dator/767609
Scraping 3/331: https://www.elgiganten.se/product/datorer-kontor/datorer/laptop/acer-chromebook-314-cel4128gb-14-barbar-dator/910407
Scraping 4/331: https://www.elgiganten.se/product/datorer-kontor/datorer/laptop/acer-chromebook-314-cel464gb-14-barbar-dator/910406
Scraping 5/331: https://www.elgiganten.se/product/datorer-kontor/datorer/laptop/acer-chromebook-314-cel8128gb-14-barbar-dator/910408
Scraping 6/331: https://www.elgiganten.se/product/datorer-kontor/datorer/laptop/acer-chromebook-314-mtk432gb-14-barbar-dator/641166
Scraping 7/331: https://www.elgiganten.se/product/datorer-kontor/datorer/laptop/acer-chromebo

In [14]:
import pandas as pd

print("Splitting specs into own individual columns so we can use ML clustering or cosine similarity on numerical & categorical vectors")
# Load the scraped Elgiganten file
df = pd.read_excel("elgiganten_laptops_full.xlsx")

# Function to extract structured specs
def parse_specs(spec_string):
    cpu = ram = storage = screen = gpu = None
    specs = spec_string.split(" | ") if isinstance(spec_string, str) else []

    for spec in specs:
        spec_lower = spec.lower()

        if "ram" in spec_lower:
            ram = spec
        elif "ssd" in spec_lower or "lagring" in spec_lower or "emmc" in spec_lower:
            storage = spec
        elif "skärm" in spec_lower or '"' in spec_lower:
            screen = spec
        elif any(brand in spec for brand in ["Intel", "AMD", "Apple"]):
            cpu = spec
        elif any(gpu_kw in spec for gpu_kw in ["RTX", "GTX", "Radeon", "Intel Iris", "MX"]):
            gpu = spec

    return pd.Series([cpu, ram, storage, screen, gpu])

# Apply parsing function to each row
df[["CPU", "RAM", "Storage", "Screen", "GPU"]] = df["Specs"].apply(parse_specs)

# Save to new Excel file
df.to_excel("elgiganten_laptops_updated.xlsx", index=False)

print("Structured file saved as 'elgiganten_laptops_updated.xlsx'")

Splitting specs into own individual columns so we can use ML clustering or cosine similarity on numerical & categorical vectors
Structured file saved as 'elgiganten_laptops_updated.xlsx'
